In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import docling
from pathlib import Path

from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from langchain_docling.loader import DoclingLoader, ExportType

from langchain_text_splitters.markdown import MarkdownHeaderTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma

In [ ]:
# Get the file path to the PDF file
FILE_PATH = Path.cwd() / "RegulationsfortheFIFAWorldCup26_noimages.pdf"

# Set your pipeline options for docling
pipeline_options = PdfPipelineOptions(
    allow_external_plugins=True,
    do_ocr=False,
    do_table_structure=True,
    force_backend_text=True,
    generate_parsed_pages=False,
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options,
            backend=PyPdfiumDocumentBackend,
        )
    }
)

loader = DoclingLoader(
    file_path=str(FILE_PATH),
    converter=converter,
    export_type=ExportType.MARKDOWN,
)

docs = loader.load()
markdown = docs[0].page_content

print("documents loaded:", len(docs))
print("markdown length:", len(markdown))

In [ ]:
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on = [("#", "Header 1"), 
                                                                ("##", "Header 2")])
pages_md_split = md_splitter.split_text(docs[0].page_content)                                                                

In [ ]:
embedding = OpenAIEmbeddings(model = "text-embedding-3-small")

In [ ]:
vectorstore = Chroma.from_documents(
    documents=pages_md_split,
    embedding=embedding,
)

print("chunks stored:", len(pages_md_split))


In [ ]:
vectorstore.get()

In [ ]:
vectorstore.get(ids = "2beb93e5-42d9-496c-ad21-d43b6be6508d", include =["embeddings"])

In [ ]:
from langchain_core.documents import Document

# 1) Add a document to the vectorstore
new_doc = Document(
    page_content=(
        "The FIFA World Cup 26 opening ceremony will take place before the first match "
    ),
    metadata={"Header 1": "ARTICLE X: OPENING CEREMONY", "source": "manual"},
)

new_doc_ids = vectorstore.add_documents([new_doc])
print("Added document id:", new_doc_ids[0])
print(vectorstore.get_by_ids(new_doc_ids))


In [ ]:
# 2) Update the document that was just added
updated_doc = Document(
    page_content=(
        "The FIFA World Cup 26 opening ceremony will take place before the first match and it will feature cultural performances from the host nations."
    ),
    metadata={"Header 1": "ARTICLE X: OPENING CEREMONY", "source": "manual", "updated": True},
)

vectorstore.update_document(document_id=new_doc_ids[0], document=updated_doc)
print("Updated document id:", new_doc_ids[0])
print(vectorstore.get_by_ids(new_doc_ids))


In [ ]:
# 3) Delete the document that was added in step 1
vectorstore.delete(ids=new_doc_ids)
print("Deleted document id:", new_doc_ids[0])
print(vectorstore.get_by_ids(new_doc_ids))
